# Diffusion Tensor Imaging Measurements for Computer-Aided Diagnosis of Autism - Derived Data from ABIDE II Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the **Diffusion Tensor Imaging Measurements for Computer-Aided Diagnosis of Autism - Derived Data from ABIDE II** dataset using the [`mlcroissant`](https://mlcommons.github.io/croissant/python/) library.

### Dataset Source
The dataset source is provided as a Croissant schema at the following URL:
`https://sen.science/doi/10.71728/senscience.dbrh-5zc8/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.dbrh-5zc8/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# You can access the metadata directly
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}\n\nDescription: {metadata.description}\n")

## 2. Data Overview
Inspect the available record sets and their fields (@id values are shown).

### Record Sets Detected
(If you see none listed below, the dataset may use the default single record set inferred from data distributions.)

In [ ]:
# List available record sets
print("Available record sets and their ID values:\n")
record_sets = []
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        print(f"- {rs['@id']} : {rs.get('name', '')}")
        record_sets.append(rs['@id'])
elif hasattr(metadata, 'distribution') and len(metadata.distribution) > 0:
    # If no explicit recordSet, infer one from distributions (this is common in simple tabular datasets)
    for dist in metadata.distribution:
        if '@id' in dist:
            rid = dist['@id']
            print(f"- {rid}")
            record_sets.append(rid)
else:
    print("No explicit record sets found in metadata.")

if not record_sets:
    # Fallback: use the default record set ID mlcroissant assigns
    default_id = None
    try:
        # Access internal data structure
        default_id = list(dataset._record_sets.keys())[0]
        record_sets = [default_id]
        print(f"Using detected record set: {default_id}")
    except Exception as e:
        print(f"Could not detect default record set: {e}")
else:
    default_id = record_sets[0]

print("\nInspect fields of the first record set:")
try:
    fields = dataset._record_sets[record_sets[0]].fields
    for field in fields:
        name = field['name'] if 'name' in field else field.get('@id', '')
        print(f"- {field['@id']}: {name}")
except Exception as e:
    print(f"Field extraction failed: {e}")

## 3. Data Extraction
Load data from the chosen record set into a DataFrame for analysis.

> **Note:** All entities are referenced by their `@id` for clarity and reproducibility.

We'll iterate over all available record sets (using their `@id`) and display the columns for each.

In [ ]:
# Prepare DataFrames for each record set
dataframes = {}
for rid in record_sets:
    print(f"Loading records for record set @id: {rid}")
    try:
        records = list(dataset.records(record_set=rid))
        df = pd.DataFrame(records)
        dataframes[rid] = df
        print(f"Record set '{rid}' fields:", df.columns.tolist())
        display(df.head())
    except Exception as e:
        print(f"Failed to load record set '{rid}': {e}")

## 4. Exploratory Data Analysis (EDA)
Demonstration of common data processing: filtering, normalizing, and grouping.

> All field/column references here use the entity `@id` (column labels in the DataFrame).
You may need to adjust the field IDs below to match exactly what was shown from the previous listing (see above DataFrame columns).

In [ ]:
from pandas.api.types import is_numeric_dtype
# Choose one DataFrame to explore (use the first found)
record_set_id = record_sets[0]
df = dataframes[record_set_id]

# Show numeric field candidates
print("Numeric fields in this record set:")
numeric_candidates = [col for col in df.columns if is_numeric_dtype(df[col])]
print(numeric_candidates)

# If none are automatically detected, try to find them using DTI metrics
if not numeric_candidates:
    # Try common DTI measurement names in the dataset
    possible_fields = [col for col in df.columns if any(x in col.lower() for x in ['fa', 'rd', 'ad', 'md', 'value'])]
    print("Possible numeric fields based on DTI metrics:")
    print(possible_fields)
    numeric_candidates = possible_fields

if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    # Fallback to the first column
    numeric_field = df.columns[0]

print(f"\nSelected field for EDA (by @id): {numeric_field}")

# Filter records (choose a threshold suitable for DTI, e.g., FA > 0.2)
threshold = 0.2
filtered_df = df[df[numeric_field] > threshold]
print(f"\nFiltered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize the field
filtered_df[numeric_field + '_normalized'] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized field '{numeric_field}' for filtered records:")
display(filtered_df[[numeric_field, numeric_field + '_normalized']].head())

# Attempt grouping by a key (try common group fields: 'ParticipantID', 'Site', etc.)
possible_groups = [col for col in df.columns if any(x in col.lower() for x in ['participant', 'subject', 'site', 'group'])]
if possible_groups:
    group_field = possible_groups[0]
    print(f"\nGrouping by field: {group_field}")
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    display(grouped_df.head())
else:
    print("\nNo common group field found for grouping.")

## 5. Visualization
Visualize distributions or relationships for selected numeric fields in the dataset.

We'll plot a histogram and a boxplot of the field explored above. Adjust the selected field if you wish to visualize others.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Histogram of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

plt.figure(figsize=(8, 4))
sns.boxplot(data=df, y=numeric_field)
plt.title(f"Boxplot of {numeric_field}")
plt.ylabel(numeric_field)
plt.show()

# If group_field is defined, visualize per-group statistics
if 'group_field' in locals():
    plt.figure(figsize=(12, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=df)
    plt.title(f"Boxplot of {numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to load and explore a real-world Croissant dataset with `mlcroissant`.

- The dataset provides DTI-derived metrics across brain regions and participants, referenced strictly by Croissant `@id`s for record sets and fields.
- Basic EDA included field filtering, normalization, and optional grouping using Croissant IDs.
- Visualizations reveal data distribution and can inform further machine learning or statistical analysis.

You can continue by applying additional statistical tests, building machine learning models, or exporting subsets. Refer to mlcroissant [documentation](https://mlcommons.github.io/croissant/python/) for advanced options.